In [1]:
import sys
import platform

print(sys.executable)
print(platform.system())

/opt/conda/bin/python
Linux


## uwin - Pipeline

เรียกใช้ฟังก์ชันต่างๆ จากโฟลเดอร์ src

In [31]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from uwinControl import config, data_io, qc, features, models, energy, finance, simulate

SITE = config.get_site() # อ่านจาก UWIN_SITE ใน docker-compose
print(SITE['station_code'])
print('เสาวัดลม:', SITE['mast_height_m'], ' ม.')
print('จังหวัด:', SITE['province'])

GWD54
เสาวัดลม: 160  ม.
จังหวัด: ยโสธร


1.ข้อมูล <br>
ข้อมูลเปลี่ยนเป็น raw = data_io.load_raw('Site_Name.csv')

In [4]:
regional = simulate.make_regional_wind('2006-01-01','2026-10-07 23:00')
era5 = simulate.make_era5(regional)
raw = simulate.make_mast_data(regional,'2025-10-08','2026-10-07 23:50')

days = (raw.index[-1] - raw.index[0]).days + 1
print('แถว/วัน = %.0f  (ควรได้ %d)' % (len(raw) / days, SITE['records_per_day']))

แถว/วัน = 144  (ควรได้ 144)


2.Freeze เวอร์ชัน ทุกการเทรนโมเดลต้องอ้างอิง version_id

In [5]:
DATA_VERSION = data_io.save_version(raw, note='snapshot ทดลอง')
raw, meta = data_io.load_version(DATA_VERSION)
print(DATA_VERSION)
data_io.list_versions()

GWD54_20260806_0718_11dde3f6


,version_id,site,created,rows,period_end,note
0,GWD54_20260806_0718_11dde3f6,GWD54,2026-08-06T07:18:14,52560,2026-10-07,snapshot ทดลอง
1,GWD54_20260806_0641_11dde3f6,GWD54,2026-08-06T06:41:18,52560,2026-10-07,snapshot ทดลอง


3.QC + Tower shadow

In [6]:
clean = qc.run_qc(raw)
display(qc.coverage_report(clean))

gap = qc.gap_structure(clean['WS100'], SITE['records_per_day'])
print('ค่าว่างที่ยาวที่สุด %.1f ชม.' % (gap['longest_gap_hours']))
print('ขาดเกิน 1 วัน %d ครั้ง' % (gap['gaps_over_1day']))

bin15 = (clean['WS160_NW'] >= 14) & (clean['WS160_NW'] < 16)
print('TI@15m/s = %.4f' % clean.loc[bin15,'TI'].mean())

,coverage_pct,status
WS160_NW,9.88,ต่ำกว่าเกณฑ์
WS160_SE,9.77,ต่ำกว่าเกณฑ์
WS140,10.11,ต่ำกว่าเกณฑ์
WS120,11.06,ต่ำกว่าเกณฑ์
WS100,11.94,ต่ำกว่าเกณฑ์
WS80,12.72,ต่ำกว่าเกณฑ์
WS60,15.13,ต่ำกว่าเกณฑ์


ค่าว่างที่ยาวที่สุด 286.2 ชม.
ขาดเกิน 1 วัน 84 ครั้ง
TI@15m/s = nan


4.Feature + wind shear (α)

In [7]:
feat = features.build_features(clean)
feat, ALPHA = features.compute_alpha(feat)
train, test = features.split_by_time(feat)
print('Alpha (α) = %.4f' % (ALPHA))
print('Train = %d , Test = %d' % (len(train), len(test)))

α = 0.1731
Train = 1845 , Test = 791


5.เทรน + วัดผล

In [13]:
print(list(trained.keys()))
print(scores.index.tolist())

['linear', 'random_forest', 'random_forest_alpha']
['Power Law (baseline)', 'Linear Regression', 'Random Forest', 'Random Forest ผ่าน α']


In [15]:
TARGET = f"WS{SITE['mast_height_m']}"
scores, trained = models.train_all(train, test, features.FEATURE_COLUMNS, TARGET, ALPHA)
display(scores)

models.save_model(trained['random_forest_alpha'], 'random_forest_alpha', DATA_VERSION, features.FEATURE_COLUMNS, scores.loc['Random Forest ผ่าน α'].to_dict())

,MAE,RMSE,MAPE,R2
model,,,,
Power Law (baseline),0.1291,0.1614,3.3025,0.8889
Linear Regression,0.0950,0.1190,2.4283,0.9396
Random Forest,0.1055,0.1355,2.6696,0.9217
Random Forest ผ่าน α,0.1025,0.1299,2.6135,0.9280


'random_forest_alpha_GWD54_20260806_0718_11dde3f6'

6.Hub height + MCP

In [26]:
HUB_HEIGHT_M = 150

all_feat = feat[features.FEATURE_COLUMNS].dropna()
wind_hub = pd.Series(all_feat['WS100'].to_numpy() * (HUB_HEIGHT_M / 100)**trained['random_forest_alpha'].predict(all_feat), index = all_feat.index)

def mcp_features(day):
    day = day.copy()
    day['month_sin'] = np.sin(2 * np.pi * day.index.dayofyear / 365)
    day['month_cos'] = np.cos(2 * np.pi * day.index.dayofyear / 365)
    day['hour_sin'] = np.sin(2 * np.pi * day.index.hour / 24)
    day['hour_cos'] = np.cos(2 * np.pi * day.index.hour / 24)
    return day

MCP_COLS = ['era_ws','month_sin','month_cos','hour_sin','hour_cos']
overlap = mcp_features(era5.join(wind_hub.resample('1h').mean().rename('site'), how = 'inner').dropna())
cut = overlap.index[int(len(overlap) * 0.7)]
mcp = models.make_random_forest(min_samples_leaf = 10).fit( overlap.loc[:cut, MCP_COLS], overlap.loc[:cut,'site'])
mcp_score = models.evaluate(overlap.loc[cut:,'site'], mcp.predict(overlap.loc[cut:, MCP_COLS]), 'MCP')
display(pd.DataFrame([mcp_score]).set_index('model').round(4))

longterm = mcp_features(era5)
longterm['wind_lt'] = mcp.predict(longterm[MCP_COLS])
annual = longterm['wind_lt'].resample('YS').mean()
IAV = float(annual.std() / annual.mean())
print('IAV = %.2f%%' % (IAV * 100))
print('ปีที่วัดมีค่าต่างจากค่าระยะยาวประมาณ %+.2f%%' % ((wind_hub.mean() / longterm['wind_lt'].mean() - 1) * 100))

,MAE,RMSE,MAPE,R2
model,,,,
MCP,0.5592,0.7028,12.0582,0.4201


IAV = 1.52%
ปีที่วัดมีค่าต่างจากค่าระยะยาวประมาณ -24.07%


7.Weibull -> AEP -> P50/P90 -> ความคุ้มค่าในการลงทุน

In [29]:
k, A = energy.fit_weibull(longterm['wind_lt'])
adc = energy.air_density(clean['Temp'].mean(), clean['Pres'].mean())
aep = energy.calculate_aep(k, A, adc)

UNCERTAINTY = {'measurement':.03, 'vertical':.04, 'mcp':.035, 'power_curve':.05, 'wake':.03, 'iav':IAV} # ค่าจำลอง
p = energy.exceedance_levels(aep['net_farm_gwh'], UNCERTAINTY)

capacity_mw = energy.DEFAULT_TURBINE['rated_kw'] * energy.DEFAULT_TURBINE['n_turbines'] / 1000
rows = []
for lab in ['P50','P75','P90']:
    r = finance.financial_analysis(p[lab], capacity_mw)
    rows.append({
        'กรณี':lab, 
        'AEP(GWh)':round(p[lab],2),
        'NPV(MUSD)':round(r['NPV_musd'],2), 
        'IRR(%)':round(r['IRR_pct'],2),
        'คืนทุน(ปี)':r['payback_years'], 
        'LCOE':round(r['LCOE_usd_per_mwh'],2)
        })

print('Weibull k = %.3f, A = %.3f ' % (k, A))
print('Air Density Correction (ρ) = %.4f' % (adc))
print('Capacity Factor = %.2f%%' % aep['capacity_factor_pct'])
print('Standard Deviation (σ) = %.2f%%' % (p['sigma_total'] * 100))
pd.DataFrame(rows)

Weibull k = 10.728, A = 4.438 
Air Density Correction (ρ) = 1.1708
Capacity Factor = 2.53%
Standard Deviation (σ) = 8.58%


,กรณี,AEP(GWh),NPV(MUSD),IRR(%),คืนทุน(ปี),LCOE
0,P50,9.98,-70.34,NaN,None,828.54
1,P75,9.40,-70.81,NaN,None,879.41
2,P90,8.88,-71.22,NaN,None,930.85


8.สรุป

In [30]:
from datetime import datetime
res90 = finance.financial_analysis(p['P90'], capacity_mw)

print('=' * 70)
print(f"ไซต์งาน {SITE['station_code']}  จังหวัด{SITE['province']}  ({SITE['latitude']}, {SITE['longitude']})")
print(f"ช่วงข้อมูล : {raw.index.min():%d %b %Y} - {raw.index.max():%d %b %Y}")
print(f"Data version : {DATA_VERSION}")
print(f"รายงาน : {datetime.now():%d %b %Y %H:%M}")
print('-'* 70)
print(f"α = {ALPHA:.4f}")
print(f"Weibull k = {k:.3f} A = {A:.3f}")
print(f"CF = {aep['capacity_factor_pct']:.2f}%")
print(f"P50 = {p['P50']:.2f}, P75 = {p['P75']:.2f}, P90 = {p['P90']:.2f} GWh/ปี")
print(f"ข้อสรุป (P90): {'คุ้มค่า' if res90['NPV_musd'] > 0 else 'ยังไม่คุ้มค่า'}")
print(f"NPV = {res90['NPV_musd']:.2f} MUSD  IRR = {res90['IRR_pct']:.2f}%")
print('=' * 70)

ไซต์งาน GWD54  จังหวัดยโสธร  (15.98565, 104.2271899)
ช่วงข้อมูล : 08 Oct 2025 - 07 Oct 2026
Data version : GWD54_20260806_0718_11dde3f6
รายงาน : 06 Aug 2026 08:23
----------------------------------------------------------------------
α = 0.1731
Weibull k = 10.728 A = 4.438
CF = 2.53%
P50 = 9.98, P75 = 9.40, P90 = 8.88 GWh/ปี
ข้อสรุป (P90): ยังไม่คุ้มค่า
NPV = -71.22 MUSD  IRR = nan%
